# Image Classifier Robustness

Bilal Abdul and Subhi Alloush · DV2607, Blekinge Institute of Technology

## Experiment

This notebook compares ResNet-50 predictions on clean and perturbed images and evaluates input preprocessing on the same examples.

### Images and baseline predictions

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

import tensorflow as tf
import keras
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image

from art.estimators.classification import KerasClassifier
from art.attacks.evasion import BoundaryAttack
from art.utils import to_categorical

tf.compat.v1.disable_eager_execution()

print(f"TensorFlow: {tf.__version__}")

In [ ]:
model = ResNet50(weights='imagenet')

In [ ]:
from pathlib import Path

image_dir = Path("images") if Path("images/koala.jpeg").is_file() else Path("../images")
init_image = image.load_img(image_dir / "koala.jpeg", target_size=(224, 224))
init_image = image.img_to_array(init_image)

target_image = image.load_img(image_dir / "tractor.jpeg", target_size=(224, 224))
target_image = image.img_to_array(target_image)

class_lookup = {105: "koala", 866: "tractor"}

In [ ]:
init_class_id = np.argmax(model.predict(np.array([init_image])))
print(f"Init image class label: {init_class_id} ({class_lookup[init_class_id]})")
plt.imshow(init_image.astype(np.uint))
plt.show()

target_class_id = np.argmax(model.predict(np.array([target_image])))
print(f"Target image class label: {target_class_id} ({class_lookup[target_class_id]})")
plt.imshow(target_image.astype(np.uint))
plt.show()

### Dependencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from art.attacks.evasion import ProjectedGradientDescent

from art.defences.preprocessor import FeatureSqueezing, JpegCompression

from tensorflow.keras.applications.resnet50 import decode_predictions

### Model and visualization helpers

In [ ]:
class GraphModeKerasClassifier(KerasClassifier):
    """ART classifier for TensorFlow graph mode."""
    def predict(self, x, batch_size=128, training_mode=False, **kwargs):
        x_prep, _ = self._apply_preprocessing(x, y=None, fit=False)
        preds = self._model.predict(x_prep, batch_size=batch_size, verbose=0)
        return self._apply_postprocessing(preds=preds, fit=False)

    def loss_gradient(self, x, y, training_mode=False, **kwargs):
        x_prep, y_prep = self._apply_preprocessing(x, y, fit=False)
        K = tf.compat.v1.keras.backend
        if not hasattr(self, "_grad_fn"):
            y_placeholder = K.placeholder(shape=(None, self.nb_classes), dtype=tf.float32)
            loss_tensor = tf.keras.losses.categorical_crossentropy(
                y_placeholder, self._output, from_logits=self._use_logits
            )
            grad_tensor = K.gradients(loss_tensor, self._input)[0]
            self._grad_fn = K.function([self._input, y_placeholder], [grad_tensor])
        grads = self._grad_fn([x_prep, y_prep])[0]
        grads = self._apply_preprocessing_gradient(x, grads)
        return grads


def make_classifier(keras_model, defenses=None):
   
    return GraphModeKerasClassifier(
        model=keras_model,
        clip_values=(0, 255),
        preprocessing_defences=defenses
    )


def show_predictions(classifier, original_image, perturbed_image, class_lookup):
    """Plot the clean image, perturbed image, and pixel differences."""
    orig_probs = classifier.predict(np.array([original_image]))[0]
    adv_probs = classifier.predict(np.array([perturbed_image]))[0]

    orig_id = int(np.argmax(orig_probs))
    adv_id = int(np.argmax(adv_probs))

    original_name = class_lookup.get(orig_id, f"Class {orig_id}")
    predicted_name = class_lookup.get(adv_id, f"Class {adv_id}")

    difference = perturbed_image - original_image
    mean_change = np.mean(np.abs(difference))
    max_change = np.max(np.abs(difference))

    print("=== IMAGE CLASSIFICATION RESULTS ===")
    print(f"Original image classified as:    {orig_id} ({original_name}) with probability {orig_probs[orig_id]*100:.2f}%")
    print(f"Perturbed image classified as: {adv_id} ({predicted_name}) with probability {adv_probs[adv_id]*100:.2f}%")
    print(f"Koala probability after perturbation: {adv_probs[105]*100:.4f}%")
    print(f"Mean absolute pixel change:   {mean_change:.2f} / 255")
    print(f"Maximum absolute pixel change:   {max_change:.2f} / 255")

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    axes[0].imshow(np.clip(original_image, 0, 255).astype(np.uint8))
    axes[0].set_title(f"Original image\nClass: {orig_id} ({original_name}, {orig_probs[orig_id]*100:.1f}%)")
    axes[0].axis("off")

    axes[1].imshow(np.clip(perturbed_image, 0, 255).astype(np.uint8))
    axes[1].set_title(f"Perturbed image\nClass: {adv_id} ({predicted_name}, {adv_probs[adv_id]*100:.1f}%)")
    axes[1].axis("off")

    noise_display = np.clip((difference - difference.min()) / (difference.max() - difference.min() + 1e-8) * 255, 0, 255)
    axes[2].imshow(noise_display.astype(np.uint8))
    axes[2].set_title(f"Perturbation (contrast enhanced)\nMaximum change: ±{max_change:.1f} per pixel")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

### Image batch

In [ ]:
x_koala = np.array([init_image])

y_target_tractor = to_categorical([target_class_id], nb_classes=1000)

print(f"Input image shape (x_koala): {x_koala.shape}")
print(f"Target class: {target_class_id} ({class_lookup[target_class_id]})")

### Baseline classifier

In [ ]:
original_classifier = make_classifier(model)

start_pred = np.argmax(original_classifier.predict(x_koala))
print(f"Baseline ART prediction: {start_pred} ({class_lookup[start_pred]})")

### Perturbed input

In [ ]:
pgd_attack = ProjectedGradientDescent(
    estimator=original_classifier,
    targeted=True,
    eps=2.5,
    eps_step=0.5,
    max_iter=15,
    verbose=False
)

adv_koala_batch = pgd_attack.generate(x=x_koala, y=y_target_tractor)
adv_koala_image = adv_koala_batch[0]

### Prediction results

The original classifier assigned 99.99% confidence to koala on the clean image and 99.99% to tractor on the perturbed image. The mean absolute pixel change was 1.53 on the 0–255 scale. The largest change was 2.50.

In [ ]:
show_predictions(original_classifier, init_image, adv_koala_image, class_lookup)

## Input preprocessing

### Evaluation helpers

In [ ]:
def evaluate_preprocessing(orig_clf, defended_clf, clean_koala, adv_koala, adaptive_koala, class_lookup):
    """Compare koala and tractor probabilities across five evaluation conditions."""
    p_original_clean = orig_clf.predict(np.array([clean_koala]))[0]
    p_orig_adv = orig_clf.predict(np.array([adv_koala]))[0]

    p_defended_clean = defended_clf.predict(np.array([clean_koala]))[0]
    p_defended_perturbed = defended_clf.predict(np.array([adv_koala]))[0]
    p_defended_adaptive = defended_clf.predict(np.array([adaptive_koala]))[0]

    cases = [
        ("1. Baseline model on clean koala", p_original_clean),
        ("2. Baseline model on perturbed image (PGD)", p_orig_adv),
        ("3. Preprocessed model on clean koala", p_defended_clean),
        ("4. Preprocessed model on perturbed image", p_defended_perturbed),
        ("5. Preprocessed model on new adaptive input", p_defended_adaptive),
    ]

    print("=== BASELINE AND PREPROCESSED MODEL ===")
    print(f"{'Test case':<46} | {'Predicted class':<20} | {'P(koala)':<10} | {'P(tractor)':<10}")
    print("-" * 96)
    for name, probs in cases:
        top_id = int(np.argmax(probs))
        top_name = class_lookup.get(top_id, f"Class {top_id}")
        print(f"{name:<46} | {top_id:>3} ({top_name:<7}, {probs[top_id]*100:>5.1f}%) | {probs[105]*100:>8.2f}% | {probs[866]*100:>8.2f}%")

    labels = [
        "Baseline model\n(Clean image)",
        "Baseline model\n(PGD input)",
        "Preprocessed model\n(Clean image)",
        "Preprocessed model\n(Perturbed image)",
        "Preprocessed model\n(Adaptive input)"
    ]
    koala_probabilities = [p[105] * 100 for _, p in cases]
    tractor_probabilities = [p[866] * 100 for _, p in cases]

    x_pos = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(11, 5))
    koala_bars = ax.bar(x_pos - width/2, koala_probabilities, width, label="Class 105: Koala (%)", color="#2ca02c")
    tractor_bars = ax.bar(x_pos + width/2, tractor_probabilities, width, label="Class 866: Tractor (%)", color="#d62728")

    ax.set_ylabel("Predicted probability (%)")
    ax.set_title("Class probabilities before and after preprocessing")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 115)
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    for bars in [koala_bars, tractor_bars]:
        for s in bars:
            height = s.get_height()
            ax.annotate(f"{height:.1f}%",
                        xy=(s.get_x() + s.get_width() / 2, height),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

### Preprocessing filters

In [ ]:
feature_squeezing = FeatureSqueezing(clip_values=(0, 255), bit_depth=4)
jpeg_filter = JpegCompression(clip_values=(0, 255), quality=60, channels_first=False)

### Classifier with preprocessing

The model applies four-bit feature squeezing and JPEG compression before inference. Its weights stay fixed.

In [ ]:
defended_classifier = make_classifier(
    keras_model=model,
    defenses=[feature_squeezing, jpeg_filter]
)

clean_koala_id = np.argmax(defended_classifier.predict(np.array([init_image])))
clean_tractor_id = np.argmax(defended_classifier.predict(np.array([target_image])))
print(f"Preprocessed model on clean koala:   {clean_koala_id} ({class_lookup[clean_koala_id]})")
print(f"Preprocessed model on clean tractor: {clean_tractor_id} ({class_lookup[clean_tractor_id]})")

adaptive_evaluation = ProjectedGradientDescent(
    estimator=defended_classifier,
    targeted=True,
    eps=2.5,
    eps_step=0.5,
    max_iter=15,
    verbose=False
)
adaptive_koala_image = adaptive_evaluation.generate(x=x_koala, y=y_target_tractor)[0]
print("Evaluation input ready.")

### Evaluation results

With preprocessing, the clean and previously perturbed koala images received koala probabilities of 99.11% and 94.90%. On a separate input perturbed against the preprocessed model, koala remained the top class at 19.80%, followed by other classes with lower probabilities. Tractor received 1.34%. These are measurements on individual images.

In [ ]:
evaluate_preprocessing(
    orig_clf=original_classifier,
    defended_clf=defended_classifier,
    clean_koala=init_image,
    adv_koala=adv_koala_image,
    adaptive_koala=adaptive_koala_image,
    class_lookup=class_lookup
)

## References

1. **Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015).** *Explaining and Harnessing Adversarial Examples.* International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1412.6572
2. **Madry, A., Makelov, A., Schmidt, L., Tsipras, D., & Vladu, A. (2018).** *Towards Deep Learning Models Resistant to Adversarial Attacks.* International Conference on Learning Representations (ICLR). https://arxiv.org/abs/1706.06083
3. **Xu, W., Evans, D., & Qi, Y. (2018).** *Feature Squeezing: Detecting Adversarial Examples in Deep Neural Networks.* Network and Distributed System Security Symposium (NDSS). https://arxiv.org/abs/1704.01155
4. **Dziugaite, G. K., Ghahramani, Z., & Roy, D. M. (2016).** *A study of the effect of JPG compression on adversarial images.* arXiv preprint. https://arxiv.org/abs/1608.00853
5. **Nicolae, M.-I., et al. (2018).** *Adversarial Robustness Toolbox v1.0.0.* arXiv:1807.01069. Documentation and source: https://github.com/Trusted-AI/adversarial-robustness-toolbox